```Gabriel A. Amici - 07/05/2026```

# Plots para o relatório 5

1. Campos otimizados e histórico funcional
    * Otimização quântica (Krotov vs MyOpt)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from matplotlib.ticker import AutoMinorLocator
from scipy.interpolate import CubicSpline
from emerald.utils import FieldParams, external_field_array

In [ ]:
plt.style.use('Ltx.mplstyle')

results_path = "./data/"
figures_path = "./figures/"


In [ ]:
results_dir = "../../../results/data/quantum_control/take_3/"

os.makedirs(figures_path, exist_ok=True)


In [ ]:
"""COLOURS"""
AmC_DEEP_BLUE = '#1d65c9'
AmC_BRIGHT_RED = '#ff1a1a'
AmC_VIVID_ORANGE = '#ff6a00'
AmC_BOLD_YELLOW = '#ffd500'
AmC_LIGHT_GREEN = '#bfe500'
AmC_LIME_GREEN = '#7ee236'

Perc7_DKBLUE_1 = '#340498'
Perc7_PURPUR_2 = '#6c00a8'
Perc7_VIOLET_3 = '#a31d99'
Perc7_PINKSH_4 = '#ca4678'
Perc7_AMBERS_5 = '#e97257'
Perc7_ORANGE_6 = '#fa9f3a'
Perc7_YELLOW_7 = '#f8db24'

Perc7 = [Perc7_DKBLUE_1, Perc7_PURPUR_2, Perc7_VIOLET_3, Perc7_PINKSH_4, Perc7_AMBERS_5, Perc7_ORANGE_6, Perc7_YELLOW_7]

Perc5_DKBLUE_1 = '#340498'
Perc5_PURPLE_2 = '#7e03a7'
Perc5_PINKSH_3 = '#b83289'
Perc5_ROSSEE_4 = '#de6063'
Perc5_ORANGE_5 = '#f8983d'

Perc5 = [Perc5_DKBLUE_1, Perc5_PURPLE_2, Perc5_PINKSH_3, Perc5_ROSSEE_4, Perc5_ORANGE_5]
Perc5_reverse = Perc5[::-1]


In [ ]:
def load_opt(filepath):
    with open(filepath, "r") as f:
        data = json.load(f)
    results = {}
    for k, v in data.get("results", {}).items():
        results[k.replace(" ", "_")] = v
    data["results"] = results
    return data

def load_opt_dir(dirpath, pattern=None):
    all_items = sorted(os.listdir(dirpath))
    out = []
    for f in all_items:
        if pattern is not None and pattern not in f:
            continue
        if f.endswith(".json"):
            full = os.path.join(dirpath, f)
            if os.path.isfile(full):
                out.append((f, load_opt(full)))
    return out

def make_time_grid(data):
    gp = data["General parameters"]
    T = gp["total_time"]
    dt = gp["delta_t"]
    return np.arange(0, T + dt, dt)

def apply_axis_style(ax, tick_len=5):
    if not ax.get_xscale() == "log":
        ax.xaxis.set_minor_locator(AutoMinorLocator())
    if not ax.get_yscale() == "log":
        ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which='both', direction='in', right=True, top=True)
    ax.tick_params(which='major', length=tick_len)
    try:
        ax.get_yticklabels()[0].set_verticalalignment("bottom")
        ax.get_yticklabels()[-1].set_verticalalignment("top")
    except (IndexError, AttributeError):
        pass
    try:
        ax.get_xticklabels()[0].set_horizontalalignment("left")
        ax.get_xticklabels()[-1].set_horizontalalignment("right")
    except (IndexError, AttributeError):
        pass


In [ ]:
all_data = load_opt_dir(results_dir)
print(f"Found {len(all_data)} files")
print()
for name, d in all_data:
    m = d["metadata"]
    r = d["results"]
    JT_key = "J_T_history" if "J_T_history" in r else "JT_history"
    jt = r[JT_key]
    print(f"{name.split('--')[0]:25s}  "
          f"n_iter={m.get('n_iterations','?'):>3}  "
          f"converged={str(m.get('converged','?')):<5}  "
          f"final_overlap={r.get('final_overlap','?'):.4f}  "
          f"best_JT={r.get('best_JT', r.get('best_J_T', '?')):.6f}  "
          f"JT_len={len(jt)}  "
          f"field_len={len(r['optimized_field'])}")


In [ ]:
pairs = {}
for name, d in all_data:
    suffix = name.split("--")[0].replace("KrotovOpt", "").replace("MyOpt", "")
    pairs.setdefault(suffix, {})
    if name.startswith("Krotov"):
        pairs[suffix]["Krotov"] = (name, d)
    else:
        pairs[suffix]["MyOpt"] = (name, d)

for k in sorted(pairs.keys()):
    v = pairs[k]
    print(f"Pair {k}: {list(v.keys())}")


In [ ]:
# --- 2x2 plot for a single pair ---
# Edit these to change what is plotted
selected_pair = "01"
pop_states = [0, 1, 2]  # which states to plot in (c) and (d)


# Load the pair
my_item = pairs.get(selected_pair, {}).get("MyOpt")
kr_item = pairs.get(selected_pair, {}).get("Krotov")
if my_item is None:
    raise ValueError(f"No MyOpt data found for pair {selected_pair}")

my_name, my_data = my_item
my_r  = my_data["results"]
my_gp = my_data["General parameters"]

time_grid = make_time_grid(my_data)

# Initial field
initial_field = np.array(my_r.get("initial_field", my_gp.get("initial_field", 0.5)))
if initial_field.ndim == 0:
    initial_field = np.full_like(time_grid, float(initial_field))

# Krotov data (if available)
if kr_item is not None:
    kr_name, kr_data = kr_item
    kr_r = kr_data["results"]
    kr_gp = kr_data["General parameters"]
    kr_field = np.array(kr_r["optimized_field"])
    kr_JT_key = "J_T_history" if "J_T_history" in kr_r else "JT_history"
    kr_JT = np.array(kr_r[kr_JT_key])

# MyOpt optimized field and JT
my_field = np.array(my_r["optimized_field"])
my_JT_key = "J_T_history" if "J_T_history" in my_r else "JT_history"
my_JT = np.array(my_r[my_JT_key])

# Populations
init_pop = np.array(my_r.get("initial_populations"))  # shape (n_states, n_times)
final_pop = np.array(my_r.get("final_populations"))  # shape (n_states, n_times)

# ---- 2x2 plot ----
fig, axes = plt.subplots(2, 2, figsize=(12, 7), dpi=148)
plt.subplots_adjust(hspace=0.25, wspace=0.2)

colors = {"MyOpt": Perc5[1], "Krotov": Perc5[3]}
ls = {"MyOpt": "solid", "Krotov": "solid"}
lw = 1.1
ax_a, ax_b, ax_c, ax_d = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

# --- (a) Top-left: initial field + optimized fields ---
ax_a.plot(time_grid, initial_field, c="gray", ls="dashed", lw=lw, markersize=0,
          label="Campo inicial")
ax_a.plot(time_grid, my_field, c=colors["MyOpt"], ls=ls["MyOpt"], lw=lw, markersize=0,
          label="Auto implementado")
if kr_item is not None:
    ax_a.plot(time_grid, kr_field, c=colors["Krotov"], ls=ls["Krotov"], lw=lw, markersize=0,
              label="Pacote krotov")
ax_a.set_title("a)", loc='left', fontweight='bold')
ax_a.set_xlabel(r"$t$")
ax_a.set_ylabel(r"$\epsilon(t)$", rotation=0, labelpad=15)
ax_a.legend(fontsize=6)
apply_axis_style(ax_a)
ax_a.set_xlim(0, time_grid[-1])

# --- (b) Top-right: convergence ---
iters = np.arange(1, len(my_JT) + 1)
ax_b.plot(iters, my_JT, c=colors["MyOpt"], ls=ls["MyOpt"], lw=lw, markersize=0,
          label="Auto implementado")
if kr_item is not None:
    kr_iters = np.arange(1, len(kr_JT) + 1)
    ax_b.plot(kr_iters, kr_JT, c=colors["Krotov"], ls=ls["Krotov"], lw=lw, markersize=0,
              label="Pacote krotov")
ax_b.set_title("b)", loc='left', fontweight='bold')
ax_b.set_ylabel(r"$J_T$", rotation=0, labelpad=15)
ax_b.set_xlabel(r"Iteração")
ax_b.set_yscale("log")
ax_b.legend(fontsize=6)
apply_axis_style(ax_b)
ax_b.set_xlim(1, None)

# --- (c) Bottom-left: initial population evolution ---
if init_pop is not None:
    for st in pop_states:
        ax_c.plot(time_grid, init_pop[st], lw=lw, markersize=0, 
                  label=rf"$\left|\varphi_{st}\right\rangle$")
ax_c.set_title("c)", loc='left', fontweight='bold')
ax_c.set_ylabel(r"População", labelpad=10)
ax_c.set_xlabel(r"$t$")
ax_c.legend(fontsize=9)
apply_axis_style(ax_c)
ax_c.set_xlim(0, time_grid[-1])
ax_c.set_ylim(0, 1)

# --- (d) Bottom-right: final population evolution ---
if final_pop is not None:
    for st in pop_states:
        ax_d.plot(time_grid, final_pop[st], lw=lw, markersize=0, 
                  label=rf"$\left|\varphi_{st}\right\rangle$")
ax_d.set_title("d)", loc='left', fontweight='bold')
ax_d.set_ylabel(r"População", labelpad=10)
ax_d.set_xlabel(r"$t$")
ax_d.legend(fontsize=9)
apply_axis_style(ax_d)
ax_d.set_xlim(0, time_grid[-1])
ax_d.set_ylim(0, 1)


In [ ]:
# --- 2x2 plot for a single pair ---
# Edit these to change what is plotted
selected_pair = "03"

# Load the pair
my_item = pairs.get(selected_pair, {}).get("MyOpt")
kr_item = pairs.get(selected_pair, {}).get("Krotov")
if my_item is None:
    raise ValueError(f"No MyOpt data found for pair {selected_pair}")

my_name, my_data = my_item
my_r  = my_data["results"]
my_gp = my_data["General parameters"]

time_grid = make_time_grid(my_data)

# Initial field
initial_field = np.array(my_r.get("initial_field", my_gp.get("initial_field", 0.5)))
if initial_field.ndim == 0:
    initial_field = np.full_like(time_grid, float(initial_field))

# Krotov data (if available)
if kr_item is not None:
    kr_name, kr_data = kr_item
    kr_r = kr_data["results"]
    kr_gp = kr_data["General parameters"]
    kr_field = np.array(kr_r["optimized_field"])
    kr_JT_key = "J_T_history" if "J_T_history" in kr_r else "JT_history"
    kr_JT = np.array(kr_r[kr_JT_key])

# MyOpt optimized field and JT
my_field = np.array(my_r["optimized_field"])
my_JT_key = "J_T_history" if "J_T_history" in my_r else "JT_history"
my_JT = np.array(my_r[my_JT_key])

# Populations
init_pop  = np.array(my_r.get("initial_populations"))  # shape (n_states, n_times)
final_pop = np.array(my_r.get("final_populations"))  # shape (n_states, n_times)

# ---- 2x2 plot ----
fig, axes = plt.subplots(2, 2, figsize=(12, 7), dpi=148)
plt.subplots_adjust(hspace=0.25, wspace=0.2)

colors = {"MyOpt": Perc5[1], "Krotov": Perc5[3]}
ls = {"MyOpt": "solid", "Krotov": "solid"}
lw = 1.1
ax_a, ax_b, ax_c, ax_d = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

# --- (a) Top-left: initial field + optimized fields ---
ax_a.plot(time_grid, initial_field, c="gray", ls="dashed", lw=lw, markersize=0,
          label="Campo inicial")
ax_a.plot(time_grid, my_field, c=colors["MyOpt"], ls=ls["MyOpt"], lw=lw, markersize=0,
          label="Auto implementado")

ax_a.set_title("a)", loc='left', fontweight='bold')
ax_a.set_xlabel(r"$t$")
ax_a.set_ylabel(r"$\epsilon(t)$", rotation=0, labelpad=15)
ax_a.legend(fontsize=6)
apply_axis_style(ax_a)
ax_a.set_xlim(0, time_grid[-1])

# --- (b) Top-right: convergence ---
iters = np.arange(1, len(my_JT) + 1)
ax_b.plot(iters, my_JT, c=colors["MyOpt"], ls=ls["MyOpt"], lw=lw, markersize=0,
          label="Auto implementado")

ax_b.set_title("b)", loc='left', fontweight='bold')
ax_b.set_ylabel(r"$J_T$", rotation=0, labelpad=15)
ax_b.set_xlabel(r"Iteração")
ax_b.set_yscale("log")
ax_b.legend(fontsize=6)
apply_axis_style(ax_b)
ax_b.set_xlim(1, None)

# --- (c) Bottom-left: initial population evolution ---
if init_pop is not None:
    bound_pop = np.sum(init_pop[:29], axis=0)
    free_pop  = np.sum(init_pop[29:], axis=0)
    ax_c.plot(time_grid, bound_pop, lw=lw, markersize=0, 
                  label="Estados ligados", c=Perc5[0])
    ax_c.plot(time_grid, free_pop, lw=lw, markersize=0, 
                  label="Estados livres", c=Perc5[3])
    
ax_c.set_title("c)", loc='left', fontweight='bold')
ax_c.set_ylabel(r"População", labelpad=10)
ax_c.set_xlabel(r"$t$")
ax_c.legend(fontsize=9)
apply_axis_style(ax_c)
ax_c.set_xlim(0, time_grid[-1])
ax_c.set_ylim(0, 1)

# --- (d) Bottom-right: final population evolution ---
if final_pop is not None:
    bound_pop = np.sum(final_pop[:29], axis=0)
    free_pop  = np.sum(final_pop[29:], axis=0)
    ax_d.plot(time_grid, bound_pop, lw=lw, markersize=0, 
                  label="Estados ligados", c=Perc5[0])
    ax_d.plot(time_grid, free_pop, lw=lw, markersize=0, 
                  label="Estados livres", c=Perc5[3])
ax_d.set_title("d)", loc='left', fontweight='bold')
ax_d.set_ylabel(r"População", labelpad=10)
ax_d.set_xlabel(r"$t$")
ax_d.legend(fontsize=9)
apply_axis_style(ax_d)
ax_d.set_xlim(0, time_grid[-1])
ax_d.set_ylim(0, 1)


In [ ]:
fig.savefig(figures_path + "QuantumIonization.pdf", bbox_inches="tight")

In [ ]:
results_dir = "../../../results/data/ionization/ionization_stability/take05/"
files = os.listdir(results_dir)

fig, ax = plt.subplots(figsize=(9, 5), dpi=148)

for i, file in enumerate(files):
    data = json.load(open((results_dir+file), 'r'))

    fields = np.array(data['parameters']['field_amplitude_set'])
    probs  = np.array(data['results']['ionization_probabilities_energy_criterion'])
    label  = f"Potencial {data['parameters']['potential']}"
    ax.plot(fields, probs, label=label, c=Perc5[i*2], markersize=0, lw=1.5)

ax.legend()
# ax.grid()
ax.set_xlim(fields[0], fields[-1])
ax.set_ylim(0, 1)
apply_axis_style(ax)
ax.set_xlabel(r"$\epsilon_0$")
ax.set_ylabel(r"$P_{ion}$", rotation=0, labelpad=10)
plt.show()



In [ ]:
np.sum(np.abs(kr_field-my_field))/len(time_grid)

In [ ]:
fig.savefig(figures_path + "Stability.pdf", bbox_inches="tight")

In [ ]:
# --- 2x2 para outro par (mude selected_pair2 para ver outro) ---
selected_pair2 = "02"

fig2, axes2 = plt.subplots(2, 2, figsize=(8, 6), dpi=148)
plt.subplots_adjust(hspace=0.3, wspace=0.35)

my2 = pairs.get(selected_pair2, {}).get("MyOpt")
kr2 = pairs.get(selected_pair2, {}).get("Krotov")
if my2 is not None:
    _, d = my2
    r = d["results"]; gp = d["General parameters"]
    tg = make_time_grid(d)
    init_f = np.array(r.get("initial_field", gp.get("initial_field", 0.5)))
    if init_f.ndim == 0:
        init_f = np.full_like(tg, float(init_f))
    my_f = np.array(r["optimized_field"])
    jtk = "J_T_history" if "J_T_history" in r else "JT_history"
    my_jt = np.array(r[jtk])
    init_p = np.array(r.get("initial_populations"))
    final_p = np.array(r.get("final_populations"))

if kr2 is not None:
    _, d2 = kr2
    r2 = d2["results"]
    kr_f = np.array(r2["optimized_field"])
    jtk2 = "J_T_history" if "J_T_history" in r2 else "JT_history"
    kr_jt = np.array(r2[jtk2])

ax2a, ax2b, ax2c, ax2d = axes2[0, 0], axes2[0, 1], axes2[1, 0], axes2[1, 1]

ax2a.plot(tg, init_f, c="gray", ls="dashed", lw=lw, label="Campo inicial")
ax2a.plot(tg, my_f, c=Perc5[1], lw=lw, label="Auto implementado")
if kr2 is not None:
    ax2a.plot(tg, kr_f, c=Perc5[3], ls="dashed", lw=lw, label="Pacote krotov")
ax2a.set_title("a)", loc='left', fontweight='bold')
ax2a.set_ylabel(r"$\epsilon(t)$", rotation=0, labelpad=15)
ax2a.legend(fontsize=6)
apply_axis_style(ax2a)
ax2a.set_xlim(0, tg[-1])

ax2b.plot(np.arange(1, len(my_jt) + 1), my_jt, c=Perc5[1], lw=lw, label="Auto implementado")
if kr2 is not None:
    ax2b.plot(np.arange(1, len(kr_jt) + 1), kr_jt, c=Perc5[3], ls="dashed", lw=lw, label="Pacote krotov")
ax2b.set_title("b)", loc='left', fontweight='bold')
ax2b.set_ylabel(r"$J_T$", rotation=0, labelpad=15)
ax2b.set_xlabel(r"iteração")
ax2b.set_yscale("log")
ax2b.legend(fontsize=6)
apply_axis_style(ax2b)
ax2b.set_xlim(1, None)

if init_p is not None:
    for st in pop_states:
        ax2c.plot(tg, init_p[st], lw=lw, label=rf"$\left|{st}\right\rangle$")
ax2c.set_title("c)", loc='left', fontweight='bold')
ax2c.set_ylabel(r"população", rotation=0, labelpad=15)
ax2c.set_xlabel(r"$t$")
ax2c.legend(fontsize=9)
apply_axis_style(ax2c)
ax2c.set_xlim(0, tg[-1]); ax2c.set_ylim(0, 1)

if final_p is not None:
    for st in pop_states:
        ax2d.plot(tg, final_p[st], lw=lw, label=rf"$\left|{st}\right\rangle$")
ax2d.set_title("d)", loc='left', fontweight='bold')
ax2d.set_ylabel(r"população", rotation=0, labelpad=15)
ax2d.set_xlabel(r"$t$")
ax2d.legend(fontsize=9)
apply_axis_style(ax2d)
ax2d.set_xlim(0, tg[-1]); ax2d.set_ylim(0, 1)

fig2.savefig(figures_path + "Fig2.pdf", bbox_inches="tight")


In [ ]:
# --- Quick-look: plot any single result file ---
# Descomente e altere o caminho para inspecionar qualquer arquivo

# custom_file = results_dir + "MyOpt03--2026-05-08T23:26:14.932588.json"
# data = load_opt(custom_file)
# r = data["results"]
# gp = data["General parameters"]
#
# print("Results keys:", list(r.keys()))
# print("GP keys:", list(gp.keys()))
# print("JT_history len:", len(r.get("JT_history", r.get("J_T_history", []))))
# print("optimized_field len:", len(r.get("optimized_field", [])))
# pop = r.get("initial_populations")
# if pop:
#     print(f"initial_populations shape: {len(pop)} states x {len(pop[0])} times")
# pop = r.get("final_populations")
# if pop:
#     print(f"final_populations shape: {len(pop)} states x {len(pop[0])} times")
